# How big are these nanoparticles? (MSc)

**Your mission.** You work in a laboratory that makes silica nanoparticles for medical imaging. A new batch must
have a radius between 4.5 and 5.5 nm: the size decides how long the particles stay in the blood and where they end
up. An electron microscope shows a few hundred dried particles; small-angle neutron scattering gives the average over
the billions of particles of the batch, in the liquid. You measure the batch on LoKI, develop the analysis, and
decide whether the batch can be released (end of part 6).

Silica nanoparticles dispersed in heavy water, measured on the virtual LoKI. Parts 1 to 4 are guided: they take your
files (sample, solvent, empty beam) to the scattering curve $I(Q)$ in absolute units, with the official ESS reduction
(`esssans`). Parts 5 and 6 are yours: **you write the Guinier analysis and the model fit**, and check cells tell you
whether your results are right (the twin knows the true sample). Part 7 is about the method.

Measure first in NICOS (empty beam, D₂O, silica spheres, same configuration; your files are in the folder of your
proposal, `f"{TWIN_RUN_ROOT}/data/loki/<year>/<proposal>/raw"`), or leave `FOLDER = None` below to start with a
demonstration session.

**The measurements in NICOS** (click *Open NICOS* in the app: NICOS of LoKI opens, ready; give a proposal
number in the *Experiment* panel, then type in the command line or the *Scripting* tab):
```
move(heavy_shutter, 'Open', fast_shutter, 'Open')
move(collimation, '3m', detector_distance, 5)
move(sample_holder, 'empty_beam')
with nexusfile_open('empty beam'):
    count(t=60)
move(sample_holder, 'd2o')
with nexusfile_open('D2O'):
    count(t=600)
move(sample_holder, 'silica_spheres')
with nexusfile_open('silica spheres'):
    count(t=600)
```
Why: the solvent is subtracted from the sample, the empty beam is the reference of the transmission; all three in the
same configuration (3 m collimation, detector at 5 m: Q from 0.006 to 0.5 Å⁻¹, enough for particles of a few nm).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))     # the modules of the LoKI twin (folder loki/)

**Your sample** (from the sample library):

In [ ]:
sys.path.insert(0, os.path.abspath("../../.."))  # the sample library (twin/sample_library.py)
import sample_library as sl, matplotlib.pyplot as plt
for name, hidden in sl.exercise_hidden("exercise.yml").items():
    spec = sl.load("loki")[name]
    print(sl.describe("loki", spec, hidden))
    sl.draw("loki", spec, hidden=hidden, fig=plt.figure(figsize=(10, 3.5))); plt.show()

## 1. Your folder

Write in `FOLDER` the folder of your measurements:
- web page, Python or console: `f"{TWIN_RUN_ROOT}/loki/<session>"`, e.g. `f"{TWIN_RUN_ROOT}/loki/student"`;
- NICOS: the folder of your proposal, `f"{TWIN_RUN_ROOT}/data/loki/<year>/<proposal>/raw"`,
  e.g. `f"{TWIN_RUN_ROOT}/data/loki/2026/3001/raw"`.

`TWIN_RUN_ROOT` is the root of the results of the twin (`/data/twin` in the container,
`C:\Users\<you>\McStas_runs\twin` on Windows). Leave `FOLDER = None` to measure first, 10 min each, an empty beam,
the solvent and the silica spheres: a few minutes of computing (~20 min the very first time: McStas simulations of
the beamline and of the samples, then kept).

In [ ]:
from loki_paths import TWIN_RUN_ROOT          # root of the results of the twins
FOLDER = None                                  # or e.g. f"{TWIN_RUN_ROOT}/data/loki/2026/3001/raw"

In [ ]:
import sys, os, json, glob, datetime, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath("../.."))
import numpy as np
import scipp as sc
import h5py
import matplotlib.pyplot as plt
from loki_paths import TWIN_RUN_ROOT
import loki_reduction as red
import loki_analysis as la
import loki_samples as ls

if FOLDER is None:                                     # demonstration session
    FOLDER = f"{TWIN_RUN_ROOT}/loki/analysis_demo"
    if not glob.glob(f"{FOLDER}/loki_*.nxs"):
        from virtual_loki import VirtualLoki
        loki = VirtualLoki(session="analysis_demo", seed=1)
        loki.move(collimation="3m", detector_distance=5.0, heavy_shutter="open", shutter="open")
        for sample, title in (("empty_beam", "empty beam"), ("d2o", "D2O, 1 mm cell"),
                              ("silica_spheres", "silica spheres in D2O")):
            loki.move(sample=sample)
            loki.count(600, title=title)
print("folder:", FOLDER)
print("files :", sorted(os.path.basename(f) for f in glob.glob(f"{FOLDER}/loki_*.nxs")))

## 2. The logbook: which file is what?

Each measurement (*count*) wrote one file `loki_000001.nxs`, `loki_000002.nxs`, ... and one line in the logbook
`log.jsonl`, with the state of the instrument during the count. A SANS experiment needs three kinds of measurement,
**in the same configuration** (collimation, pulse skipping, detector distance):
- **sample**: the particles in their solvent, in a cell;
- **solvent** (the "background"): the same cell with the solvent only. Its scattering is subtracted from the sample's,
  which leaves the scattering of the particles alone;
- **empty beam**: nothing in the beam. The transmission monitor behind the sample position then counts the full beam:
  the reference for the transmission of the sample and of the solvent (the fraction of neutrons that cross them).

In [ ]:
rows = []
with open(f"{FOLDER}/log.jsonl", encoding="utf-8") as f:
    for line in f:
        entry = json.loads(line)
        d = entry["details"]
        if entry["action"] != "count" or not d.get("file"):
            continue
        s = d["state"]
        rows.append(dict(file=os.path.basename(d["file"]), kind=red.kind(s), title=d["title"], t_s=d["t"],
                         collimation=s["collimation"], pulse_skipping=s.get("pulse_skipping", "off"),
                         detector_m=s["detector_distance"], sample=s["sample"]))
def show_table(rows):                                  # a list of dicts, printed as a table
    if not rows:
        print("no measurement in the logbook"); return
    cols = list(rows[0])
    width = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in cols}
    print("  ".join(c.ljust(width[c]) for c in cols))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(width[c]) for c in cols))

show_table(rows)

The cell below takes the **last** sample measurement and the last solvent and empty beam measured in the same
configuration. To choose other files, write their names, e.g. `SAMPLE = "loki_000003.nxs"`.

In [ ]:
def config(r):
    return (r["collimation"], r["pulse_skipping"], r["detector_m"])

samples = [r for r in rows if r["kind"] == "sample"]
SAMPLE = samples[-1]["file"] if samples else None
same = [r for r in rows if SAMPLE and config(r) == config(samples[-1])]
last = lambda k: ([r["file"] for r in same if r["kind"] == k] or [None])[-1]
SOLVENT, EMPTY = last("solvent"), last("empty beam")
print(f"sample: {SAMPLE}   solvent: {SOLVENT}   empty beam: {EMPTY}")
if None in (SAMPLE, SOLVENT, EMPTY):
    print("A sample, a solvent and an empty beam measured in the same configuration are needed: measure the missing "
          "one(s), then run this notebook again.")
path = lambda name: f"{FOLDER}/{name}" if name else None

## 3. Inside a NeXus file

The files have the format of the real LoKI at ESS: the twin starts from a file written by the ESS file writer at LoKI
and replaces the measured data by its own. LoKI has **9 detector banks** (`loki_detector_0` is the rear bank, 1-4 the
middle banks, 5-8 the front banks), made of tubes, each tube of 7 thin "straws", in 4 layers: about 2.8 million
pixels. Like ODIN, the detectors record **each neutron** as an *event*:
- `event_id`: the pixel hit (its position is in the file: `x_pixel_offset`, ...);
- `event_time_offset`: the arrival time, counted from the start of its source pulse (14 pulses per second);
- `event_time_zero` / `event_index`: when each pulse happened and which events belong to it.

The **monitors** (`beam_monitor_m1` before the sample, `beam_monitor_m3` after it) count the beam itself: they say how
many neutrons of each wavelength were sent, and how many crossed the sample.

In [ ]:
with h5py.File(path(SAMPLE), "r") as f:
    e = f["entry"]
    txt = lambda x: x.decode() if isinstance(x, bytes) else x
    print("title       :", txt(e["title"][()]), "  sample:", txt(e["sample/name"][()]))
    print("start / end :", txt(e["start_time"][()]), "/", txt(e["end_time"][()]))
    det = e["instrument/loki_detector_0"]
    ev = det["detector_events"]
    t_zero, index = ev["event_time_zero"][:3], ev["event_index"][:3]
    ids, toa = ev["event_id"][:5], ev["event_time_offset"][:5]
    numbers = det["detector_number"][()]
    pos = np.stack([det[c][()] for c in ("x_pixel_offset", "y_pixel_offset", "z_pixel_offset")], 1)
    print("\nrear bank: source pulses")
    print("  pulse   event_time_zero [ns]   = date and time (UTC)            event_index")
    for k, (tz, i0) in enumerate(zip(t_zero, index)):
        date = datetime.datetime.fromtimestamp(tz / 1e9, datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S.%f")
        print(f"  {k:5d}   {tz:20d}   {date}   {i0:10d}")
    print("\nrear bank: first neutrons")
    print("  event   event_id (pixel)   pixel position relative to the bank [mm]   event_time_offset [ms]")
    for k, (pix, t) in enumerate(zip(ids, toa)):
        x, y, z = pos[np.searchsorted(numbers, pix)] * 1e3
        print(f"  {k:5d}   {pix:16d}   ({x:+7.1f}, {y:+7.1f}, {z:+6.1f})              {t / 1e6:9.3f}")
    print("\nevents per bank:", {b: len(e[f"instrument/{b}/detector_events/event_id"]) for b in red.BANK_GROUPS["rear"]
                                 + red.BANK_GROUPS["middle"] + red.BANK_GROUPS["front"]})
    print("monitor events:", {m: len(e[f"instrument/{m}/monitor_events/event_time_offset"])
                              for m in ("beam_monitor_m1", "beam_monitor_m3")})
    tw = e["twin"]
    print(f"\nfraction of the neutrons written as events: {tw['event_sampling'][()]:.2e}")

**Reading the extract.** Each neutron is a pixel number and an arrival time, nothing more: its scattering angle comes
from the position of the pixel, its wavelength from the arrival time. The pulses are 71.4 ms apart (1/14 s).

**Why only a fraction of the neutrons?** LoKI counts millions of neutrons per second; a 10-minute measurement would
give tens of GB of events. The twin writes a random sample of them (the fraction above), the **same fraction** for the
detectors and the monitors, so everything normalized by the monitors (the intensity in absolute units, the
transmission) is unchanged; only the counting noise is that of the events in the file. The group `entry/twin` holds
this fraction and the truth about the sample: it does not exist in a real file.

## 4. Reduction with the ESS software: $I(Q)$

**The scattering vector $Q$.** A neutron of wavelength $\lambda$ deflected by an angle $2\theta$ has
$Q = 4\pi\sin\theta/\lambda$ (Å⁻¹). A structure of size $D$ scatters mostly around $Q \approx 2\pi/D$.

The ESS workflow (`esssans`, `LokiWorkflow`, the same code as for the real LoKI) does, for each bank:
1. **arrival time → wavelength**, with a lookup table of the chopper configuration (built from the McStas beam);
2. **pixel → angle → $Q$**, from the pixel positions in the file (with the small fall of the neutrons by gravity);
3. **normalization** by the incident monitor: per wavelength, how many neutrons were sent;
4. **transmission** $T(\lambda)$: the transmission monitor of the sample (or solvent) run divided by that of the empty
   beam, each normalized by its incident monitor;
5. **solid angle** of each pixel times its efficiency (the **flat field**: not all pixels detect equally well; the
   instrument team measures it with a flat scatterer), and the **direct-beam function** $D(\lambda)$, the calibration
   that makes the result absolute (measured with a standard of known cross-section);
6. **background subtraction**: $I_\text{sample} - I_\text{solvent}$, each normalized as above.

The result, divided by the thickness of the sample (1 mm), is the differential scattering cross-section per unit
volume $d\Sigma/d\Omega$ in cm⁻¹ sr⁻¹: the fraction of neutrons scattered per cm of sample and per unit of solid angle.
It does not depend on the instrument or the counting time. About 1 min for the 9 banks (a few minutes the first time
for a configuration: the twin measures its calibration).

In [ ]:
BANKS = [f"loki_detector_{i}" for i in range(9)]
iq = red.reduce(path(SAMPLE), background=path(SOLVENT), empty_beam=path(EMPTY), banks=BANKS)
thickness_cm = float(red.lnx.read_twin(path(SAMPLE))["sample_thickness_m"]) * 100
groups = red.combine(iq, thickness_cm)                 # rear, middle, front: banks merged

COLORS = {"rear": "tab:blue", "middle": "tab:green", "front": "tab:red"}
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for g, (q, I, dI) in groups.items():
    ax[0].errorbar(q, I, dI, fmt="o", ms=3, lw=0.8, color=COLORS[g], label=f"{g} bank(s)")
ax[0].set(xscale="log", yscale="log", xlabel="Q [Å⁻¹]", ylabel="dΣ/dΩ [cm⁻¹ sr⁻¹]",
          title=f"{SAMPLE} − {SOLVENT}: scattering of the particles")
ax[0].legend(); ax[0].grid(alpha=0.3, which="both")

from ess.sans.types import Filename, SampleRun, TransmissionRun, EmptyBeamRun, TransmissionFraction
wf = red.workflow(path(SAMPLE))
wf[Filename[SampleRun]] = wf[Filename[TransmissionRun[SampleRun]]] = path(SAMPLE)
wf[Filename[EmptyBeamRun]] = path(EMPTY)
tf = wf.compute(TransmissionFraction[SampleRun])
ax[1].plot(tf.coords["wavelength"].values if tf.coords["wavelength"].sizes == tf.sizes else
           sc.midpoints(tf.coords["wavelength"]).values, tf.values, "k.")
ax[1].set(xlabel="wavelength λ [Å]", ylabel="transmission T", ylim=(0.5, 1.2), title="transmission of the sample")
ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

# from now on: the three groups of banks together, sorted by Q
q = np.concatenate([v[0] for v in groups.values()]); I = np.concatenate([v[1] for v in groups.values()])
dI = np.concatenate([v[2] for v in groups.values()])
o = np.argsort(q); q, I, dI = q[o], I[o], dI[o]

**Reading the curve.** The three groups of banks join up: they measure the same sample at different angles. From low
to high $Q$:
- a **plateau** at low $Q$: at these large distances each particle scatters like a point; its height $I(0)$ is
  proportional to the number of particles and to the square of their volume;
- a **steep fall** with **oscillations**: the minima are the signature of particles that all have the same size
  (here spheres); their position gives the size;
- at high $Q$ the curve goes down to **zero** within the error bars: the flat scattering of the solvent has been
  subtracted (without the solvent run it would stay at about 0.05 cm⁻¹).

The transmission (right) is close to 1: a 1 mm layer of D₂O with 1 % of silica lets almost all neutrons through. Its
points scatter by a few %: the monitors count few neutrons. Count the minima on the left, and note their $Q$.

## 5. Your turn: Guinier analysis

At small $Q$ the scattering of any particle follows $I(Q) \simeq I(0)\exp(-Q^2R_g^2/3)$, valid for $Q R_g \lesssim 1.3$.

**Task 5.1.** Turn this law into a straight line: what do you plot against what? Make the plot with the arrays `q`,
`I`, `dI` of part 4, for the lowest $Q$.

**Task 5.2.** Fit the line with weights (the uncertainty of $\ln I$ is $dI/I$), in the range $Q R_g < 1.3$. The range
depends on $R_g$, which you do not know yet: how do you proceed? Store $R_g$, its uncertainty and $I(0)$ (cm⁻¹).

In [ ]:
rg, drg, i0 = None, None, None     # your results [Å, Å, cm⁻¹]

**Check.** If it is wrong, read the explanation and try again; the solution is behind the button (opening it is noted in your results).

In [ ]:
import loki_exercises as ex, quiz
q_ = quiz.Quiz("loki_silica_msc")
ex.check_guinier(rg, drg, i0)
q_.solution("tasks 5.1 and 5.2", explanation="$\\ln I = \\ln I(0) - (R_g^2/3)\\,Q^2$: a straight line in $Q^2$ "
            "with slope $-R_g^2/3$. The valid range depends on $R_g$: start from a guess, fit, recompute the range "
            "$Q R_g < 1.3$, and repeat until $R_g$ no longer changes.", code=chr(10).join([
    'rg = 40.0                                                # a first guess [Å], improved below',
    'for _ in range(6):',
    '    m = (q * rg < 1.3) & (I > 0) & np.isfinite(dI) & (dI > 0)',
    '    p, cov = np.polyfit(q[m] ** 2, np.log(I[m]), 1, w=I[m] / dI[m], cov="unscaled")   # w = 1 / sigma(ln I)',
    '    rg = np.sqrt(-3 * p[0])',
    'drg = 1.5 * np.sqrt(cov[0, 0]) / rg                      # rg = sqrt(-3 s): d rg = 1.5 ds / rg',
    'i0 = np.exp(p[1])']))

## 6. Your turn: a model fit with sasmodels (SasView)

`sasmodels` is the computing engine of SasView. Its `sphere` model gives $d\Sigma/d\Omega$ in cm⁻¹, with the
parameters `scale` (the volume fraction $\phi$), `radius` (Å), `sld` and `sld_solvent` (scattering length densities,
$10^{-6}$ Å⁻²: silica 3.47, D₂O 6.36) and `background` (cm⁻¹). How to compute it:

```python
from sasmodels.core import load_model
from sasmodels.direct_model import call_kernel
model = load_model("sphere")
kernel = model.make_kernel([q])
Iq = call_kernel(kernel, dict(scale=0.01, radius=50, sld=3.47, sld_solvent=6.36, background=0))
```

**Task 6.1.** Fit $\phi$, $R$ and a background to the whole curve (e.g. with `scipy.optimize.least_squares` on the
residuals $(I_\text{model} - I)/dI$). Use your Guinier result for the starting radius. Store the results and their
uncertainties (from the Jacobian: $\text{cov} = (J^TJ)^{-1}$).

In [ ]:
radius, dradius, phi, dphi = None, None, None, None     # your results [Å, Å, -, -]

**Check.**

In [ ]:
ex.check_sphere(radius, dradius, phi, dphi)
q_.solution("task 6.1", explanation="Residuals weighted by the error bars; the covariance of the parameters is "
            "$(J^TJ)^{-1}$ with the Jacobian of these residuals. `x_scale` tells the fit the size of each parameter.",
            code=chr(10).join([
    'from scipy.optimize import least_squares',
    'from sasmodels.core import load_model',
    'from sasmodels.direct_model import call_kernel',
    'm = np.isfinite(I) & np.isfinite(dI) & (dI > 0)',
    'kernel = load_model("sphere").make_kernel([q[m]])',
    '',
    'def residuals(x):',
    '    phi_, r_, bg_ = x',
    '    model = call_kernel(kernel, dict(scale=phi_, radius=r_, sld=3.47, sld_solvent=6.36, background=bg_))',
    '    return (model - I[m]) / dI[m]',
    '',
    'fit = least_squares(residuals, [0.01, rg * np.sqrt(5 / 3), 0.0], x_scale=[0.01, 10.0, 0.01])',
    'cov = np.linalg.inv(fit.jac.T @ fit.jac)',
    'phi, radius = fit.x[0], fit.x[1]',
    'dphi, dradius = np.sqrt(cov[0, 0]), np.sqrt(cov[1, 1])']))

**Your verdict.**

In [ ]:
q_.choice("The batch must have a radius between 4.5 and 5.5 nm. With your results, what do you write?",
          ["Released: R = 5.0 nm, within the specification (its uncertainty is far smaller than the margin).",
           "Rejected: R = 3.9 nm.", "Rejected: R = 50 nm.", "Cannot be decided: the uncertainty is too large."],
          answer=0, why={1: "3.9 nm is Rg; for a sphere R = √(5/3)·Rg ≈ 5.0 nm.",
                         2: "50 Å = 5.0 nm (1 nm = 10 Å).",
                         3: "Your uncertainty on R is a fraction of an Å, the margin is ±5 Å."},
          explanation="R ≈ 50 Å = 5.0 nm, in the 4.5-5.5 nm window, with an uncertainty much smaller than the margin.",
          title="Verdict")

## 7. Developing the method

1. **Background.** Reduce the sample without subtracting the solvent (`red.reduce(path(SAMPLE), empty_beam=path(EMPTY))`)
   and fit again with and without a `background` parameter. What goes wrong, and why does it matter most for $R_g$?
2. **Resolution.** Plot your residuals: where is the model worse than the data, and why? Look at the `dq` (resolution)
   option of `make_kernel`/`call_kernel` in the sasmodels documentation: what would you need to know about the
   instrument to use it?
3. **Configuration.** Particles 4 times larger: which configuration of LoKI (collimation, detector distance) would you
   choose, and what would it cost in counting time? Check with the twin.
4. **Statistics.** Which banks dominate the uncertainty on $R$? On $\phi$? Fit with the rear bank only to see it.

Then check your understanding:

In [ ]:
q_.choice("A new batch has particles 4 times larger (R = 200 Å, Rg = 155 Å). The Guinier range needs Q < 1.3/Rg. "
          "Which configuration of LoKI do you choose?",
          ["8 m collimation, detector at 10 m (rear bank from 0.0082 Å⁻¹), and count longer: fewer neutrons.",
           "3 m collimation, detector at 5 m: the most neutrons.",
           "The same as for the 5 nm particles.",
           "Pulse skipping with the detector at 5 m: it reaches the lowest Q."],
          answer=0, why={1: "It starts at 0.0115 Å⁻¹, above 1.3/155 = 0.0084 Å⁻¹: no Guinier range left.",
                         2: "The same configuration starts at 0.0115 Å⁻¹, above the 0.0084 Å⁻¹ needed.",
                         3: "Pulse skipping at 5 m starts at 0.0106 Å⁻¹: better, but still above 0.0084 Å⁻¹."},
          explanation="1.3/155 Å = 0.0084 Å⁻¹: only the 8 m collimation with the detector at 10 m reaches it (0.0082 "
                      "Å⁻¹), at the edge. Its beam is more parallel, with fewer neutrons: count longer.",
          title="Larger particles")
q_.report()